In [246]:
import torch
import plotly.graph_objects as go
import os
from plotly.subplots import make_subplots
from typing import Literal
import plotly.colors as pc
import sys
from pathlib import Path
import math
import pandas as pd

# Ensure the project root is on sys.path so the `src` package can be imported from a notebook.
# Assumes this notebook lives in a subfolder of the project (e.g. /path/to/project/visualization).
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.functions import get_elbow_index, get_top_k, get_logprobs_diff_elbow

# Utils

In [247]:
langs = [
    "fra_Latn",
    "eng_Latn",
    "por_Latn",
    "spa_Latn",
    "jpn_Jpan",
    "zho_Hans",
    "swh_Latn",
    "wol_Latn",
    "hin_Deva",
    "arb_Arab",
    "rus_Cyrl",
]

In [248]:
models = [
    "gemma-3-4b-pt",
    "gemma-3-1b-pt",
    "gemma-3-270m",
]

In [249]:
def style_fig(fig: go.Figure) -> go.Figure:
    fig.update_layout(
        # template="plotly_white",
        font=dict(family="Arial", size=14),
        legend=dict(
            title="", orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1
        ),
        margin=dict(l=50, r=50, t=50, b=50),
    )
    fig.update_xaxes(
        title_font=dict(size=16, family="Arial"), tickfont=dict(size=14, family="Arial")
    )
    fig.update_yaxes(
        title_font=dict(size=16, family="Arial"), tickfont=dict(size=14, family="Arial")
    )
    return fig

In [250]:
def load_logprobs_diff(model, source, target, type: Literal["lang", "trad"]):
    filename = (
        f"../results/translation_task/logprobs_diff_{type}/{model}:{source}:{target}.pt"
    )
    if os.path.exists(filename):
        return torch.load(filename, map_location=torch.device("cpu"))
    else:
        raise FileNotFoundError(
            f"No logprobs_diff file found for {model} with source {source} and target {target}."
        )


def load_all_logprobs_diff(model: str, langs: list[str], type: Literal["lang", "trad"]):
    logprobs_diff_dict = {}
    for source in langs:
        for target in langs:
            if source != target:
                try:
                    logprobs_diff = load_logprobs_diff(model, source, target, type)
                    logprobs_diff_dict[(source, target)] = logprobs_diff
                except FileNotFoundError:
                    continue

    return logprobs_diff_dict

In [ ]:
def plot_logprobs_diff(
    logprobs_diffs: dict[tuple[str, str], torch.Tensor],
    langs: list[tuple[str, str]],
    title: str,
):
    cols = 3
    rows = math.ceil(len(langs) / cols) + 1

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[
            "Overall Mean",
            "Mean (eng_Latn as Source)",
            "Mean (eng_Latn as Target)",
        ]
        + [
            f"{source.split('_')[0]} -> {target.split('_')[0]} (Mean)"
            for source, target in langs
        ],
    )

    mean_logprobs_diff_all = torch.stack(
        [logprobs_diffs[(source, target)].mean(dim=-1) for source, target in langs],
        dim=0,
    ).mean(dim=0)

    max_logprobs_diff_all = (
        torch.stack(
            [logprobs_diffs[(source, target)].mean(dim=-1) for source, target in langs],
            dim=0,
        )
        .max()
        .item()
    )

    mean_logprobs_diff_source = torch.stack(
        [
            logprobs_diffs[(source, target)].mean(dim=-1)
            for source, target in langs
            if source == "eng_Latn"
        ],
        dim=0,
    ).mean(dim=0)

    mean_logprobs_diff_target = torch.stack(
        [
            logprobs_diffs[(source, target)].mean(dim=-1)
            for source, target in langs
            if target == "eng_Latn"
        ],
        dim=0,
    ).mean(dim=0)

    fig.add_trace(
        go.Heatmap(
            z=mean_logprobs_diff_all.numpy(),
            x=[f"H{i}" for i in range(mean_logprobs_diff_all.shape[1])],
            y=[f"L{i}" for i in range(mean_logprobs_diff_all.shape[0])],
            colorscale="RdBu",
            zmid=0.0,
            zmax=max_logprobs_diff_all,
            zmin=-max_logprobs_diff_all,
            showscale=False,
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Heatmap(
            z=mean_logprobs_diff_source.numpy(),
            x=[f"H{i}" for i in range(mean_logprobs_diff_source.shape[1])],
            y=[f"L{i}" for i in range(mean_logprobs_diff_source.shape[0])],
            colorscale="RdBu",
            zmid=0.0,
            zmax=max_logprobs_diff_all,
            zmin=-max_logprobs_diff_all,
            showscale=False,
        ),
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Heatmap(
            z=mean_logprobs_diff_target.numpy(),
            x=[f"H{i}" for i in range(mean_logprobs_diff_target.shape[1])],
            y=[f"L{i}" for i in range(mean_logprobs_diff_target.shape[0])],
            colorscale="RdBu",
            zmid=0.0,
            zmax=max_logprobs_diff_all,
            zmin=-max_logprobs_diff_all,
            showscale=False,
        ),
        row=1,
        col=3,
    )

    for i, (source, target) in enumerate(langs):
        mean_logprobs_diff = logprobs_diffs[(source, target)].mean(dim=-1)

        fig.add_trace(
            go.Heatmap(
                z=mean_logprobs_diff.numpy(),
                x=[f"H{i}" for i in range(mean_logprobs_diff.shape[1])],
                y=[f"L{i}" for i in range(mean_logprobs_diff.shape[0])],
                colorscale="RdBu",
                zmid=0.0,
                zmax=max_logprobs_diff_all,
                zmin=-max_logprobs_diff_all,
                showscale=False,
            ),
            row=(i // cols) + 2,
            col=(i % cols) + 1,
        )

    fig.update_layout(height=400 * rows, width=1200, title_text=title)

    fig = style_fig(fig)

    fig.show()

In [252]:
def plot_logprobs_diff_contribution(
    logprobs_diffs: dict[tuple[str, str], torch.Tensor],
    langs: list[tuple[str, str]],
    title: str,
):
    contributions = {}
    for source, target in langs:
        logprobs_diff, _ = torch.sort(
            logprobs_diffs[(source, target)].mean(dim=-1).clamp(min=0).flatten(),
            descending=True,
        )
        contributions[(source, target)] = torch.cumsum(
            logprobs_diff / logprobs_diff.sum(), dim=0
        )

    # Make one line plot with all contributions
    fig = go.Figure()
    for i, ((source, target), contrib) in enumerate(contributions.items()):
        fig.add_trace(
            go.Scatter(
                y=contrib.numpy(),
                mode="lines",
                name=f"{source.split('_')[0]} -> {target.split('_')[0]}",
                line=dict(color=pc.qualitative.Plotly[i % len(pc.qualitative.Plotly)]),
            )
        )

    mean_logprobs_diff, _ = torch.sort(
        torch.stack(
            [logprobs_diffs[(source, target)].mean(dim=-1) for source, target in langs]
        )
        .mean(dim=0)
        .clamp(min=0)
        .flatten(),
        descending=True,
    )
    contributions_mean = torch.cumsum(
        mean_logprobs_diff / mean_logprobs_diff.sum(), dim=0
    )

    fig.add_trace(
        go.Scatter(
            y=contributions_mean.numpy(),
            mode="lines",
            name="Mean over all",
            line=dict(width=3, dash="dash", color="Black"),
        )
    )

    elbow_index = get_elbow_index(contributions_mean.numpy())

    fig.add_vline(
        x=elbow_index,
        line=dict(color="Red", dash="dash"),
        annotation_text="Elbow Point",
        annotation_position="top right",
    )

    fig.update_layout(
        title=title,
        xaxis_title="Number of top contributing attention scores",
        yaxis_title="Cumulative contribution to logprobs diff",
    )

    fig = style_fig(fig)

    fig.show()

In [253]:
def plot_top_k_heads(
    logprobs_diffs: dict[tuple[str, str], torch.Tensor],
    langs: list[tuple[str, str]],
    top_k: int,
    title: str,
):
    # Heatmap plot with the number of times each head is in the top_k contributing heads across language pairs
    n_layers = logprobs_diffs[next(iter(logprobs_diffs))].shape[0]
    n_heads = logprobs_diffs[next(iter(logprobs_diffs))].shape[1]

    heads_count_en_source = torch.zeros((n_layers, n_heads), dtype=torch.int32)
    heads_count_en_target = torch.zeros((n_layers, n_heads), dtype=torch.int32)
    heads_count = torch.zeros((n_layers, n_heads), dtype=torch.int32)

    for source, target in langs:
        top_heads_indices = get_top_k(
            logprobs_diffs[(source, target)].mean(dim=-1), top_k
        )
        for layer, head in top_heads_indices:
            heads_count[layer, head] += 1

            if source == "eng_Latn":
                heads_count_en_source[layer, head] += 1
            if target == "eng_Latn":
                heads_count_en_target[layer, head] += 1

    fig = make_subplots(
        rows=2,
        cols=2,
        specs=[
            [{"colspan": 2}, None],  # First plot spans both columns
            [{}, {}],  # Second row has two independent subplots
        ],
        subplot_titles=[
            f"Overall Top-K({top_k}) Heads Count",
            "Top-K Heads Count (English as Source)",
            "Top-K Heads Count (English as Target)",
        ],
    )

    fig.add_trace(
        go.Heatmap(
            z=heads_count.numpy(),
            x=[f"Head {i}" for i in range(heads_count.shape[1])],
            y=[f"Layer {i}" for i in range(heads_count.shape[0])],
            colorscale="Viridis",
            zmin=0,
            zmax=len(langs),
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Heatmap(
            z=heads_count_en_source.numpy(),
            x=[f"Head {i}" for i in range(heads_count_en_source.shape[1])],
            y=[f"Layer {i}" for i in range(heads_count_en_source.shape[0])],
            colorscale="Viridis",
            zmin=0,
            zmax=len(langs),
        ),
        row=2,
        col=1,
    )

    fig.add_trace(
        go.Heatmap(
            z=heads_count_en_target.numpy(),
            x=[f"Head {i}" for i in range(heads_count_en_target.shape[1])],
            y=[f"Layer {i}" for i in range(heads_count_en_target.shape[0])],
            colorscale="Viridis",
            zmin=0,
            zmax=len(langs),
        ),
        row=2,
        col=2,
    )

    fig.update_layout(
        title=title,
        xaxis_title="Heads",
        yaxis_title="Layers",
        width=800,
        height=800,
    )

    fig = style_fig(fig)

    fig.show()

In [254]:
def load_scores(
    model: str, score_type: Literal["bleu", "chrf"]
) -> tuple[
    dict[tuple[str, str], float],
    dict[tuple[str, str], dict[tuple[int, int], float]],
]:
    generation_path = Path("../results/translation_task/generations/")
    baseline_score, intervention_scores = {}, {}

    for source in langs:
        for target in langs:
            for lang_head in range(6):
                for trad_head in range(6):
                    filename = (
                        generation_path
                        / f"{model}:{source}:{target}:{source}:{target}:{lang_head}:{trad_head}.csv"
                    )
                    if filename.exists():
                        if (source, target) not in intervention_scores:
                            intervention_scores[(source, target)] = {}

                        df = pd.read_csv(filename)

                        baseline_score[(source, target)] = df[
                            f"{score_type}_baseline"
                        ].mean()
                        intervention_scores[(source, target)][
                            (lang_head, trad_head)
                        ] = df[f"{score_type}_function_vector"].mean()

In [255]:
def plot_scores(
    scores: dict[tuple[str, str], dict[tuple[int, int], float]], title: str
):
    cols = 3
    rows = math.ceil(len(scores) / cols) + 1

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=["", "Overall Mean", ""]
        + [
            f"{source.split('_')[0]} -> {target.split('_')[0]}"
            for source, target in scores.keys()
        ],
    )

# Langs

## Gemma-3-4b-pt

In [256]:
logprobs_diff_lang = load_all_logprobs_diff("gemma-3-4b-pt", langs, type="lang")

In [257]:
plot_logprobs_diff(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Traduction gemma-3-4b-pt",
)

In [258]:
plot_logprobs_diff_contribution(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Contribution Language gemma-3-4b-pt",
)

In [259]:
plot_top_k_heads(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    top_k=get_logprobs_diff_elbow(logprobs_diff_lang, logprobs_diff_lang.keys()),
    title="Top 10 Heads Contribution Language gemma-3-4b-pt",
)

## Gemma-3-1b-pt

In [260]:
logprobs_diff_lang = load_all_logprobs_diff("gemma-3-1b-pt", langs, type="lang")

In [261]:
plot_logprobs_diff(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Language gemma-3-1b-pt",
)

In [262]:
plot_logprobs_diff_contribution(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Contribution Language gemma-3-1b-pt",
)

In [263]:
plot_top_k_heads(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    top_k=get_logprobs_diff_elbow(logprobs_diff_lang, logprobs_diff_lang.keys()),
    title="Top 10 Heads Contribution Language gemma-3-1b-pt",
)

## Gemma-3-270m-pt

In [264]:
logprobs_diff_lang = load_all_logprobs_diff("gemma-3-270m", langs, type="lang")

In [265]:
plot_logprobs_diff(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Language gemma-3-270m-pt",
)

In [266]:
plot_logprobs_diff_contribution(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    title="Logprobs Diff Contribution Language gemma-3-270m-pt",
)

In [267]:
plot_top_k_heads(
    logprobs_diff_lang,
    logprobs_diff_lang.keys(),
    top_k=get_logprobs_diff_elbow(logprobs_diff_lang, logprobs_diff_lang.keys()),
    title="Top 10 Heads Contribution Language gemma-3-270m-pt",
)

# Traduction

## Gemma-3-4b-pt

In [268]:
logprobs_diff_trad = load_all_logprobs_diff("gemma-3-4b-pt", langs, type="trad")

In [269]:
plot_logprobs_diff(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Traduction gemma-3-4b-pt",
)

In [270]:
plot_logprobs_diff_contribution(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Contribution Traduction gemma-3-4b-pt",
)

In [271]:
plot_top_k_heads(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    top_k=get_logprobs_diff_elbow(logprobs_diff_trad, logprobs_diff_trad.keys()),
    title="Top Heads Contribution Language gemma-3-4b-pt",
)

## Gemma-3-1b-pt

In [272]:
logprobs_diff_trad = load_all_logprobs_diff("gemma-3-1b-pt", langs, type="trad")

In [273]:
plot_logprobs_diff(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Traduction gemma-3-1b-pt",
)

In [274]:
plot_logprobs_diff_contribution(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Contribution Traduction gemma-3-1b-pt",
)

In [275]:
plot_top_k_heads(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    top_k=get_logprobs_diff_elbow(logprobs_diff_trad, logprobs_diff_trad.keys()),
    title="Top Heads Contribution Language gemma-3-1b-pt",
)

## Gemma-3-270m-pt

In [276]:
logprobs_diff_trad = load_all_logprobs_diff("gemma-3-270m", langs, type="trad")

In [278]:
plot_logprobs_diff(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Traduction gemma-3-270m-pt",
)

In [279]:
plot_logprobs_diff_contribution(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    title="Logprobs Diff Contribution Traduction gemma-3-270m-pt",
)

In [280]:
plot_top_k_heads(
    logprobs_diff_trad,
    logprobs_diff_trad.keys(),
    top_k=get_logprobs_diff_elbow(logprobs_diff_trad, logprobs_diff_trad.keys()),
    title="Top Heads Contribution Language gemma-3-270m",
)